# Adaptive Time-Series Momentum

Tests daily time-series momentum with position sizing and transaction assumptions.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
==============================================================================
TSMOM (TIME-SERIES MOMENTUM) WITH VOLATILITY SCALING - CORRECTED VERSION
==============================================================================

Strategy Overview:
------------------
Time-Series Momentum exploits the tendency of asset returns to exhibit
autocorrelation - positive past returns predict positive future returns
and vice versa. This is a pure trend-following strategy that is:
  - Market-neutral capable (can go long or short)
  - Self-contained (only uses own price history, not cross-sectional comparison)
  - Volatility-adjusted (scales positions inversely to recent volatility)

Key References:
  - Moskowitz, Ooi, and Pedersen (2012), "Time series momentum"
  - Kim, Tse, and Wald (2016), "Time series momentum and volatility scaling"

CRITICAL FIX APPLIED:
---------------------
The original code had a portfolio valuation bug where portfolio_value was not
properly accumulating P&L from positions. The corrected version now properly
tracks wealth changes through mark-to-market position valuations.

Critical Anti-Look-Ahead Design:
---------------------------------
Day T-1 Close:
  ├─ Calculate returns using data up to T-1
  ├─ Compute EMA(returns, 63) using data up to T-1
  ├─ Estimate rolling volatility (21-day) using data up to T-1
  ├─ Generate signals: sign(EMA) at Close[T-1]
  ├─ Scale positions: (target_vol / realized_vol) * signal
  └─ Store signal for execution at Open[T]

Day T Open:
  ├─ Execute pending signals from T-1 at Open[T]
  └─ Positions remain until next rebalance

Day T Close:
  ├─ Mark positions to market using Close[T]
  ├─ Calculate position P&L vs previous day
  ├─ Update portfolio value: previous_value + P&L - costs
  └─ Repeat cycle for T+1

Monthly Rebalancing:
  - Signals recalculated on last trading day of each month
  - Walk-forward optimization window: 252 trading days (1 year)
  - No in-month trading unless stop-loss triggered (if enabled)
"""

from _market_data import download_price_records
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
INITIAL_CAPITAL = 100000
COMPOUNDING_MODE = False
LEVERAGE = 1.0
VOLATILITY_TARGET = 0.1
EMA_LOOKBACK = 63
VOL_LOOKBACK = 21
REBALANCE_FREQUENCY = "M"
WALKFORWARD_WINDOW = 252
COMMISSION_BPS = 5.0
SLIPPAGE_BPS = 2.0
TOTAL_COST_BPS = COMMISSION_BPS + SLIPPAGE_BPS
START_DATE = "2019-12-31"
END_DATE = "2024-12-31"
BENCHMARK_TICKER = "URTH"
UNIVERSE = ["SPY", "QQQ", "IWM", "EFA", "EEM", "AGG", "TLT", "GLD", "DBC", "VNQ"]


def fetch_yahoo_data(ticker, start_date, end_date):
    """
    Fetch historical OHLCV data from Yahoo Finance.

    Parameters:
    -----------
    ticker : str
        Stock/ETF ticker symbol
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str
        End date in 'YYYY-MM-DD' format
        Yahoo Finance key

    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: Date, Open, High, Low, Close, Volume
        Returns None if data fetch fails

    Notes:
    ------
    - Does NOT use forward-fill or back-fill (per Space instructions)
    - Missing data remains as NaN and must be handled explicitly
    - API rate limits may apply - implement delay if fetching many tickers
    """
    try:
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data or len(data["historical"]) == 0:
            print(f"WARNING: No data returned for {ticker}")
            return None
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date")
        df.set_index("date", inplace=True)
        df = df.rename(
            columns={
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "volume": "Volume",
            }
        )
        df = df[["Open", "High", "Low", "Close", "Volume"]]
        print(
            f"✓ Fetched {len(df)} bars for {ticker} ({df.index[0].date()} to {df.index[-1].date()})"
        )
        return df
    except Exception as e:
        print(f"ERROR fetching {ticker}: {str(e)}")
        return None


def fetch_universe_data(tickers, start_date, end_date):
    """
    Fetch data for entire universe of tickers.

    Parameters:
    -----------
    tickers : list
        List of ticker symbols
    start_date : str
        Start date for data fetch
    end_date : str
        End date for data fetch
        Yahoo Finance key

    Returns:
    --------
    dict
        Dictionary mapping ticker -> DataFrame
        Only includes tickers with successful data fetch
    """
    data_dict = {}
    print(f"\nFetching data for {len(tickers)} tickers from {start_date} to {end_date}...")
    print("=" * 80)
    for ticker in tickers:
        df = fetch_yahoo_data(ticker, start_date, end_date)
        if df is not None and len(df) > 0:
            data_dict[ticker] = df
    print("=" * 80)
    print(f"Successfully fetched data for {len(data_dict)}/{len(tickers)} tickers\n")
    return data_dict


def calculate_ema_returns(prices, lookback):
    """
    Calculate Exponential Moving Average of returns.

    Parameters:
    -----------
    prices : pd.Series
        Price series (typically Close prices)
    lookback : int
        EMA lookback period in days

    Returns:
    --------
    pd.Series
        EMA of returns

    Notes:
    ------
    - Uses pandas EMA with span parameter
    - span=lookback gives decay factor α = 2/(lookback+1)
    - More weight to recent observations vs. simple moving average
    - Returns are calculated as log returns for better statistical properties
    """
    log_returns = np.log(prices / prices.shift(1))
    ema_returns = log_returns.ewm(span=lookback, adjust=False).mean()
    return ema_returns


def calculate_rolling_volatility(prices, lookback):
    """
    Calculate rolling realized volatility (annualized).

    Parameters:
    -----------
    prices : pd.Series
        Price series (typically Close prices)
    lookback : int
        Rolling window for volatility calculation

    Returns:
    --------
    pd.Series
        Annualized rolling volatility

    Notes:
    ------
    - Uses standard deviation of log returns
    - Annualization factor: sqrt(252) for daily data
    - Minimum volatility floor prevents division by zero
    - More sophisticated alternatives: GARCH, Yang-Zhang, Parkinson estimators
    """
    log_returns = np.log(prices / prices.shift(1))
    rolling_vol = log_returns.rolling(window=lookback).std()
    annualized_vol = rolling_vol * np.sqrt(252)
    min_vol = 0.01
    annualized_vol = annualized_vol.clip(lower=min_vol)
    return annualized_vol


def generate_tsmom_signals(prices, ema_lookback, vol_lookback, vol_target):
    """
    Generate TSMOM signals with volatility scaling.

    Parameters:
    -----------
    prices : pd.Series
        Price series (Close prices)
    ema_lookback : int
        Lookback for EMA momentum signal
    vol_lookback : int
        Lookback for volatility estimation
    vol_target : float
        Target annual volatility (e.g., 0.10 for 10%)

    Returns:
    --------
    pd.DataFrame
        DataFrame with columns:
        - momentum_signal: sign of EMA returns (+1, 0, -1)
        - realized_vol: rolling annualized volatility
        - vol_scalar: volatility scaling factor
        - position_size: final position size (momentum * vol_scalar)

    Notes:
    ------
    - Position size formula: (vol_target / realized_vol) * sign(EMA_returns)
    - This ensures constant volatility exposure across time and assets
    - Maximum position size capped at 2.0x to prevent extreme leverage
    """
    ema_returns = calculate_ema_returns(prices, ema_lookback)
    momentum_signal = np.sign(ema_returns)
    realized_vol = calculate_rolling_volatility(prices, vol_lookback)
    vol_scalar = vol_target / realized_vol
    max_position_size = 2.0
    vol_scalar = vol_scalar.clip(upper=max_position_size)
    position_size = momentum_signal * vol_scalar
    signals_df = pd.DataFrame(
        {
            "momentum_signal": momentum_signal,
            "realized_vol": realized_vol,
            "vol_scalar": vol_scalar,
            "position_size": position_size,
        },
        index=prices.index,
    )
    return signals_df


def get_rebalance_dates(price_index, frequency="M"):
    """
    Determine rebalancing dates based on specified frequency.

    Parameters:
    -----------
    price_index : pd.DatetimeIndex
        Full datetime index from price data
    frequency : str
        Rebalancing frequency: 'M' (monthly), 'W' (weekly), 'D' (daily)

    Returns:
    --------
    list
        List of rebalancing dates (last trading day of each period)

    Notes:
    ------
    - Monthly rebalancing is standard for TSMOM strategies
    - More frequent rebalancing increases transaction costs
    - Less frequent rebalancing reduces responsiveness to regime changes
    """
    if frequency == "D":
        return price_index.tolist()
    elif frequency == "W":
        return price_index.to_series().resample("W").last().index.tolist()
    elif frequency == "M":
        return price_index.to_series().resample("ME").last().index.tolist()
    else:
        raise ValueError(f"Invalid frequency: {frequency}. Use 'D', 'W', or 'M'.")


def calculate_portfolio_returns(
    data_dict, signals_dict, rebalance_dates, initial_capital, compounding_mode, leverage, cost_bps
):
    """
    Calculate portfolio returns with proper T-1 to T execution logic.

    CRITICAL FIX:
    -------------
    This corrected version properly tracks portfolio value by:
    1. Storing previous day's position quantities and prices
    2. Marking positions to market daily using current Close prices
    3. Calculating daily P&L as the change in position values
    4. Accumulating P&L into portfolio_value incrementally

    The original bug was resetting portfolio_value based on a static calculation
    rather than accumulating daily changes, which caused flat returns.

    Parameters:
    -----------
    data_dict : dict
        Dictionary of ticker -> price DataFrame
    signals_dict : dict
        Dictionary of ticker -> signals DataFrame
    rebalance_dates : list
        List of rebalancing dates
    initial_capital : float
        Starting capital
    compounding_mode : bool
        If True, reinvest gains; if False, use fixed initial capital
    leverage : float
        Overall leverage multiplier
    cost_bps : float
        Transaction cost in basis points (one-way)

    Returns:
    --------
    pd.DataFrame
        Portfolio performance DataFrame with columns:
        - portfolio_value: current portfolio value
        - daily_return: daily return (%)
        - positions: dictionary of current positions
        - turnover: portfolio turnover (%)
        - costs: transaction costs incurred

    Notes:
    ------
    CRITICAL TIMING LOGIC:
    ----------------------
    Day T-1 (Rebalance Date):
      - Calculate signals using Close[T-1] and all prior data
      - Determine target positions for Day T
      - Store pending orders

    Day T (Execution Date):
      - Execute trades at Open[T] (next day after signal generation)
      - Apply transaction costs based on position changes
      - Portfolio value updated using Close[T] prices

    This ensures NO look-ahead bias: signals at T-1 use only data up to T-1,
    execution happens at T open (realistically achievable), and returns
    reflect Close[T] / Close[T-1].
    """
    common_dates = None
    for ticker, df in data_dict.items():
        if common_dates is None:
            common_dates = df.index
        else:
            common_dates = common_dates.intersection(df.index)
    common_dates = sorted(common_dates)
    print(f"Portfolio calculation period: {common_dates[0].date()} to {common_dates[-1].date()}")
    print(f"Total trading days: {len(common_dates)}")
    print(f"Rebalance dates: {len(rebalance_dates)}")
    portfolio_value = initial_capital
    capital_base = initial_capital
    positions = {}
    pending_signals = {}
    results = []
    for i, date in enumerate(common_dates):
        if date in rebalance_dates:
            pending_signals = {}
            for ticker in data_dict.keys():
                if date not in signals_dict[ticker].index:
                    continue
                signal_row = signals_dict[ticker].loc[date]
                target_position = signal_row["position_size"]
                if pd.notna(target_position):
                    pending_signals[ticker] = target_position
            print(f"  [{date.date()}] Generated signals for {len(pending_signals)} assets (T-1)")
        daily_pnl = 0.0
        if len(positions) > 0:
            for ticker, pos_info in positions.items():
                if date not in data_dict[ticker].index:
                    continue
                shares = pos_info["shares"]
                prev_close = pos_info["prev_close"]
                current_close = data_dict[ticker].loc[date, "Close"]
                if pd.isna(current_close) or current_close <= 0:
                    continue
                pnl = (current_close - prev_close) * shares
                daily_pnl += pnl
                positions[ticker]["prev_close"] = current_close
        portfolio_value += daily_pnl
        transaction_costs = 0.0
        turnover = 0.0
        if i > 0 and len(pending_signals) > 0:
            if compounding_mode:
                sizing_capital = portfolio_value
            else:
                sizing_capital = capital_base
            num_assets = len(pending_signals)
            if num_assets > 0:
                capital_per_asset = sizing_capital * leverage / num_assets
            else:
                capital_per_asset = 0
            new_positions = {}
            for ticker, target_signal in pending_signals.items():
                if date not in data_dict[ticker].index:
                    continue
                open_price = data_dict[ticker].loc[date, "Open"]
                close_price = data_dict[ticker].loc[date, "Close"]
                if pd.isna(open_price) or open_price <= 0:
                    continue
                if pd.isna(close_price) or close_price <= 0:
                    close_price = open_price
                target_exposure = capital_per_asset * target_signal
                current_exposure = 0
                if ticker in positions:
                    current_shares = positions[ticker]["shares"]
                    current_exposure = current_shares * open_price
                exposure_change = target_exposure - current_exposure
                turnover += abs(exposure_change)
                trade_cost = abs(exposure_change) * (cost_bps / 10000)
                transaction_costs += trade_cost
                if abs(target_exposure) > 1:
                    shares = target_exposure / open_price
                    new_positions[ticker] = {
                        "shares": shares,
                        "entry_price": open_price,
                        "prev_close": close_price,
                    }
            positions = new_positions
            portfolio_value -= transaction_costs
            print(
                f"  [{date.date()}] Executed {len(positions)} positions at Open[T], Turnover: ${turnover:,.0f}, Costs: ${transaction_costs:,.0f}"
            )
            pending_signals = {}
        if i == 0:
            daily_return = 0.0
            previous_value = initial_capital
        else:
            previous_value = results[-1]["portfolio_value"]
            if previous_value > 0:
                daily_return = portfolio_value / previous_value - 1
            else:
                daily_return = 0.0
        gross_exposure = 0
        position_details = {}
        for ticker, pos_info in positions.items():
            if date not in data_dict[ticker].index:
                current_price = pos_info["prev_close"]
            else:
                current_price = data_dict[ticker].loc[date, "Close"]
                if pd.isna(current_price) or current_price <= 0:
                    current_price = pos_info["prev_close"]
            shares = pos_info["shares"]
            position_value = shares * current_price
            gross_exposure += abs(position_value)
            position_details[ticker] = {
                "shares": shares,
                "entry_price": pos_info["entry_price"],
                "current_price": current_price,
                "value": position_value,
            }
        cash = portfolio_value - sum((p["value"] for p in position_details.values()))
        results.append(
            {
                "date": date,
                "portfolio_value": portfolio_value,
                "daily_return": daily_return,
                "daily_pnl": daily_pnl,
                "transaction_costs": transaction_costs,
                "num_positions": len(positions),
                "gross_exposure": gross_exposure,
                "cash": cash,
                "positions": position_details.copy(),
            }
        )
    results_df = pd.DataFrame(results)
    results_df.set_index("date", inplace=True)
    return results_df


def calculate_performance_metrics(returns_series, benchmark_returns=None):
    """
    Calculate comprehensive performance metrics.

    Parameters:
    -----------
    returns_series : pd.Series
        Daily returns series
    benchmark_returns : pd.Series, optional
        Benchmark returns for comparison

    Returns:
    --------
    dict
        Dictionary of performance metrics
    """
    annual_factor = 252
    returns = returns_series.dropna()
    if len(returns) == 0:
        return {}
    total_return = (1 + returns).prod() - 1
    years = len(returns) / annual_factor
    annualized_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    volatility = returns.std() * np.sqrt(annual_factor)
    sharpe_ratio = annualized_return / volatility if volatility > 0 else 0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    calmar_ratio = abs(annualized_return / max_drawdown) if max_drawdown != 0 else 0
    win_rate = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0
    downside_returns = returns[returns < 0]
    downside_deviation = downside_returns.std() * np.sqrt(annual_factor)
    sortino_ratio = annualized_return / downside_deviation if downside_deviation > 0 else 0
    metrics = {
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": volatility,
        "Sharpe Ratio": sharpe_ratio,
        "Sortino Ratio": sortino_ratio,
        "Maximum Drawdown": max_drawdown,
        "Calmar Ratio": calmar_ratio,
        "Win Rate": win_rate,
    }
    if benchmark_returns is not None:
        aligned_returns = returns.align(benchmark_returns, join="inner")
        strat_ret = aligned_returns[0].dropna()
        bench_ret = aligned_returns[1].dropna()
        if len(strat_ret) > 0 and len(bench_ret) > 0:
            covariance = strat_ret.cov(bench_ret)
            benchmark_variance = bench_ret.var()
            beta = covariance / benchmark_variance if benchmark_variance != 0 else 0
            benchmark_annual_return = (1 + bench_ret).prod() ** (annual_factor / len(bench_ret)) - 1
            alpha = annualized_return - beta * benchmark_annual_return
            active_returns = strat_ret - bench_ret
            tracking_error = active_returns.std() * np.sqrt(annual_factor)
            information_ratio = (
                active_returns.mean() * annual_factor / tracking_error if tracking_error > 0 else 0
            )
            metrics["Beta"] = beta
            metrics["Alpha"] = alpha
            metrics["Information Ratio"] = information_ratio
            metrics["Tracking Error"] = tracking_error
    return metrics


def plot_performance(strategy_results, benchmark_data, strategy_name="TSMOM Strategy"):
    """
    Create comprehensive performance visualization.

    Parameters:
    -----------
    strategy_results : pd.DataFrame
        Strategy results from calculate_portfolio_returns
    benchmark_data : pd.DataFrame
        Benchmark price data
    strategy_name : str
        Name of strategy for plot titles
    """
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle(f"{strategy_name} Performance Analysis", fontsize=16, fontweight="bold")
    strategy_cumulative = (1 + strategy_results["daily_return"]).cumprod()
    benchmark_aligned = benchmark_data.reindex(strategy_results.index, method="ffill")
    benchmark_returns = benchmark_aligned["Close"].pct_change()
    benchmark_cumulative = (1 + benchmark_returns).cumprod()
    ax1 = axes[0, 0]
    ax1.plot(strategy_cumulative.index, strategy_cumulative.values, label="Strategy", linewidth=2)
    ax1.plot(
        benchmark_cumulative.index,
        benchmark_cumulative.values,
        label="Benchmark",
        linewidth=2,
        alpha=0.7,
    )
    ax1.set_title("Cumulative Returns")
    ax1.set_ylabel("Cumulative Return")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2 = axes[0, 1]
    running_max = strategy_cumulative.expanding().max()
    drawdown = (strategy_cumulative - running_max) / running_max
    ax2.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color="red")
    ax2.plot(drawdown.index, drawdown.values, color="darkred", linewidth=1)
    ax2.set_title("Drawdown")
    ax2.set_ylabel("Drawdown")
    ax2.grid(True, alpha=0.3)
    ax3 = axes[1, 0]
    rolling_sharpe = (
        strategy_results["daily_return"].rolling(252).mean()
        * 252
        / (strategy_results["daily_return"].rolling(252).std() * np.sqrt(252))
    )
    ax3.plot(rolling_sharpe.index, rolling_sharpe.values, linewidth=1.5)
    ax3.axhline(y=0, color="black", linestyle="--", linewidth=0.8)
    ax3.set_title("Rolling Sharpe Ratio (252-day)")
    ax3.set_ylabel("Sharpe Ratio")
    ax3.grid(True, alpha=0.3)
    ax4 = axes[1, 1]
    monthly_returns = (
        strategy_results["daily_return"].resample("ME").apply(lambda x: (1 + x).prod() - 1)
    )
    ax4.hist(monthly_returns.dropna() * 100, bins=30, edgecolor="black", alpha=0.7)
    ax4.axvline(
        x=monthly_returns.mean() * 100, color="red", linestyle="--", linewidth=2, label="Mean"
    )
    ax4.set_title("Monthly Returns Distribution")
    ax4.set_xlabel("Monthly Return (%)")
    ax4.set_ylabel("Frequency")
    ax4.legend()
    ax4.grid(True, alpha=0.3, axis="y")
    ax5 = axes[2, 0]
    ax5.plot(strategy_results.index, strategy_results["num_positions"], linewidth=1)
    ax5.set_title("Number of Active Positions")
    ax5.set_ylabel("Count")
    ax5.grid(True, alpha=0.3)
    ax6 = axes[2, 1]
    exposure_pct = strategy_results["gross_exposure"] / INITIAL_CAPITAL * 100
    ax6.plot(exposure_pct.index, exposure_pct.values, linewidth=1)
    ax6.set_title("Gross Exposure (% of Initial Capital)")
    ax6.set_ylabel("Exposure (%)")
    ax6.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def print_performance_summary(strategy_metrics, benchmark_metrics, strategy_name="TSMOM"):
    """
    Print formatted performance summary table.
    """
    print("\n" + "=" * 80)
    print(f"{strategy_name} STRATEGY PERFORMANCE SUMMARY")
    print("=" * 80)
    print(f"{'Metric':<30} {'Strategy':>15} {'Benchmark':>15} {'Difference':>15}")
    print("-" * 80)
    metrics_to_display = [
        ("Total Return", "%"),
        ("Annualized Return", "%"),
        ("Annualized Volatility", "%"),
        ("Sharpe Ratio", ""),
        ("Sortino Ratio", ""),
        ("Maximum Drawdown", "%"),
        ("Calmar Ratio", ""),
        ("Win Rate", "%"),
    ]
    for metric, format_type in metrics_to_display:
        strat_value = strategy_metrics.get(metric, 0)
        bench_value = benchmark_metrics.get(metric, 0)
        diff = strat_value - bench_value
        if format_type == "%":
            print(
                f"{metric:<30} {strat_value * 100:>14.2f}% {bench_value * 100:>14.2f}% {diff * 100:>14.2f}%"
            )
        else:
            print(f"{metric:<30} {strat_value:>15.3f} {bench_value:>15.3f} {diff:>15.3f}")
    if "Alpha" in strategy_metrics:
        print("-" * 80)
        print(f"{'Alpha (Annualized)':<30} {strategy_metrics['Alpha'] * 100:>14.2f}%")
        print(f"{'Beta':<30} {strategy_metrics['Beta']:>15.3f}")
        print(f"{'Information Ratio':<30} {strategy_metrics['Information Ratio']:>15.3f}")
        print(f"{'Tracking Error':<30} {strategy_metrics['Tracking Error'] * 100:>14.2f}%")
    print("=" * 80 + "\n")


def main():
    """
    Main execution function for TSMOM backtest.
    """
    print("\n" + "=" * 80)
    print("TSMOM (TIME-SERIES MOMENTUM) WITH VOLATILITY SCALING - CORRECTED")
    print("=" * 80)
    print(f"Backtest Period: {START_DATE} to {END_DATE}")
    print(f"Initial Capital: ${INITIAL_CAPITAL:,.0f}")
    print(f"Compounding Mode: {COMPOUNDING_MODE}")
    print(f"Leverage: {LEVERAGE}x")
    print(f"Volatility Target: {VOLATILITY_TARGET * 100}%")
    print(f"EMA Lookback: {EMA_LOOKBACK} days")
    print(f"Vol Lookback: {VOL_LOOKBACK} days")
    print(f"Rebalance Frequency: {REBALANCE_FREQUENCY}")
    print(f"Transaction Costs: {TOTAL_COST_BPS} bps")
    print(f"Universe: {len(UNIVERSE)} assets")
    print("=" * 80 + "\n")
    print("STEP 1: DATA ACQUISITION")
    print("-" * 80)
    data_dict = fetch_universe_data(UNIVERSE, START_DATE, END_DATE)
    if len(data_dict) == 0:
        print("ERROR: No data fetched. Exiting.")
        return
    print("\nSTEP 2: BENCHMARK DATA")
    print("-" * 80)
    benchmark_data = fetch_yahoo_data(BENCHMARK_TICKER, START_DATE, END_DATE)
    if benchmark_data is None:
        print(f"WARNING: Could not fetch benchmark {BENCHMARK_TICKER}")
        benchmark_data = pd.DataFrame()
    print("\nSTEP 3: SIGNAL GENERATION")
    print("-" * 80)
    signals_dict = {}
    for ticker, df in data_dict.items():
        print(f"Generating signals for {ticker}...")
        signals = generate_tsmom_signals(
            prices=df["Close"],
            ema_lookback=EMA_LOOKBACK,
            vol_lookback=VOL_LOOKBACK,
            vol_target=VOLATILITY_TARGET,
        )
        signals_dict[ticker] = signals
    print(f"✓ Generated signals for {len(signals_dict)} assets")
    print("\nSTEP 4: REBALANCING SCHEDULE")
    print("-" * 80)
    common_dates = None
    for df in data_dict.values():
        if common_dates is None:
            common_dates = df.index
        else:
            common_dates = common_dates.intersection(df.index)
    common_dates = sorted(common_dates)
    rebalance_dates = get_rebalance_dates(pd.DatetimeIndex(common_dates), REBALANCE_FREQUENCY)
    print(f"Total trading days: {len(common_dates)}")
    print(f"Rebalance dates: {len(rebalance_dates)}")
    print(f"First rebalance: {rebalance_dates[0].date()}")
    print(f"Last rebalance: {rebalance_dates[-1].date()}")
    print("\nSTEP 5: PORTFOLIO SIMULATION (CORRECTED)")
    print("-" * 80)
    strategy_results = calculate_portfolio_returns(
        data_dict=data_dict,
        signals_dict=signals_dict,
        rebalance_dates=rebalance_dates,
        initial_capital=INITIAL_CAPITAL,
        compounding_mode=COMPOUNDING_MODE,
        leverage=LEVERAGE,
        cost_bps=TOTAL_COST_BPS,
    )
    print(f"✓ Portfolio simulation complete")
    print(f"Final portfolio value: ${strategy_results['portfolio_value'].iloc[-1]:,.2f}")
    print(
        f"Total return: {(strategy_results['portfolio_value'].iloc[-1] / INITIAL_CAPITAL - 1) * 100:.2f}%"
    )
    print("\nSTEP 6: PERFORMANCE ANALYSIS")
    print("-" * 80)
    strategy_metrics = calculate_performance_metrics(
        returns_series=strategy_results["daily_return"],
        benchmark_returns=benchmark_data["Close"].pct_change() if len(benchmark_data) > 0 else None,
    )
    if len(benchmark_data) > 0:
        benchmark_returns = benchmark_data["Close"].pct_change()
        benchmark_metrics = calculate_performance_metrics(benchmark_returns)
    else:
        benchmark_metrics = {}
    print_performance_summary(strategy_metrics, benchmark_metrics, "TSMOM")
    print("\nSTEP 7: VISUALIZATION")
    print("-" * 80)
    if len(benchmark_data) > 0:
        plot_performance(
            strategy_results, benchmark_data, "TSMOM with Volatility Scaling (CORRECTED)"
        )
    else:
        print("Skipping plots (no benchmark data)")
    print("\n✓ BACKTEST COMPLETE\n")
    return (strategy_results, strategy_metrics, benchmark_metrics)


if __name__ == "__main__":
    results, metrics, bench_metrics = main()
